[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/26_recommender_systems/02_collaborative_filtering.ipynb)

# Bài 02 — Collaborative filtering dựa trên láng giềng

**🎯 Mục tiêu**
- Ý tưởng collaborative filtering (CF): dùng **hành vi của người khác**, không cần đặc trưng nội dung.
- **Baseline predictor** $b_{ui} = \mu + b_u + b_i$ (có regularization) from scratch — thành phần quan trọng nhất cho RMSE.
- Độ tương đồng: cosine, cosine trên phần dư (≈ Pearson), **shrinkage**.
- **Item-based** và **user-based KNN** cho dự đoán rating; KNN trên **phản hồi ngầm** cho top-N ("người xem X cũng xem Y").
- Vấn đề thực tế: thưa, cold start, mở rộng quy mô.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/26_recommender_systems

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ml_course.metrics import ndcg_at_k, precision_at_k, recall_at_k
from ml_course.visualization import set_style

set_style()

In [ ]:
ratings = pd.read_csv("../../data/sample/ratings.csv")
items = pd.read_csv("../../data/sample/items.csv")
n_users, n_items = ratings.user_id.max() + 1, ratings.item_id.max() + 1


def split_per_user(ratings, test_frac=0.2, min_ratings=5, seed=0):
    rng = np.random.default_rng(seed)
    test_mask = np.zeros(len(ratings), bool)
    for _, g in ratings.groupby("user_id"):
        if len(g) >= min_ratings:
            test_mask[rng.choice(g.index, max(1, int(test_frac * len(g))), replace=False)] = True
    return ratings[~test_mask], ratings[test_mask]


train, test = split_per_user(ratings)                                   # cùng cách chia với bài 01
R = np.full((n_users, n_items), np.nan)
R[train.user_id, train.item_id] = train.rating
observed = ~np.isnan(R)
u_tr, i_tr, y_tr = train.user_id.to_numpy(), train.item_id.to_numpy(), train.rating.to_numpy(float)
u_te, i_te, y_te = test.user_id.to_numpy(), test.item_id.to_numpy(), test.rating.to_numpy(float)


def rmse(pred):
    return float(np.sqrt(np.mean((y_te - np.clip(pred, 1, 5)) ** 2)))

## 1. Ý tưởng

> Nếu An và Bình đã đồng ý với nhau về nhiều phim trong quá khứ, An có thể thích phim Bình thích mà An chưa xem.

CF chỉ cần ma trận user–item: không cần biết phim **là gì**. Hai cách nhìn:
- **User-based**: tìm user giống u, gợi ý phim họ thích.
- **Item-based**: tìm phim giống phim u đã thích — "giống" = được **cùng những người** đánh giá tương tự.

## 2. Baseline predictor: thiên lệch user và item

Nhiều biến động trong rating là **thiên lệch** đơn giản: có người chấm dễ, có phim hay hơn mặt bằng. Mô hình
$$\hat r_{ui} = \mu + b_u + b_i, \qquad \min_{b}\sum_{(u,i)}(r_{ui} - \mu - b_u - b_i)^2 + \lambda\Big(\sum_u b_u^2 + \sum_i b_i^2\Big)$$
Giải luân phiên (cố định $b_u$ → nghiệm đóng cho $b_i$ và ngược lại):
$$b_i = \frac{\sum_{u \in U_i}(r_{ui} - \mu - b_u)}{\lambda + |U_i|}, \qquad b_u = \frac{\sum_{i \in I_u}(r_{ui} - \mu - b_i)}{\lambda + |I_u|}$$
$\lambda$ co thiên lệch của user/item **ít dữ liệu** về 0 — cùng tinh thần trung bình giảm chấn ở bài 01.

In [ ]:
mu = y_tr.mean()


def fit_biases(lam=5.0, n_iter=15):
    b_u, b_i = np.zeros(n_users), np.zeros(n_items)
    for _ in range(n_iter):
        b_i = np.bincount(i_tr, y_tr - mu - b_u[u_tr], n_items) / (lam + np.bincount(i_tr, minlength=n_items))
        b_u = np.bincount(u_tr, y_tr - mu - b_i[i_tr], n_users) / (lam + np.bincount(u_tr, minlength=n_users))
    return b_u, b_i


b_u, b_i = fit_biases()
rmse_table = {"trung bình toàn cục μ": rmse(np.full(len(y_te), mu)),
              "μ + b_u + b_i": rmse(mu + b_u[u_te] + b_i[i_te])}
pd.Series(rmse_table, name="RMSE test").round(4)

Chỉ hai vector thiên lệch đã giảm RMSE từ ~0.98 xuống ~0.63. Mọi mô hình CF phía sau học **phần dư** $r_{ui} - b_{ui}$.

## 3. Độ tương đồng

Với hai item $i, j$, so sánh hai cột phần dư $\mathbf z_i, \mathbf z_j$ (0 ở ô trống):
$$\text{sim}(i, j) = \frac{\mathbf z_i^\top\mathbf z_j}{\lVert\mathbf z_i\rVert\,\lVert\mathbf z_j\rVert}\cdot\frac{n_{ij}}{n_{ij} + \beta}$$
- Dùng **phần dư** (đã trừ thiên lệch) thay vì rating thô: "cả hai đều 4 sao" không có nghĩa gì nếu user đó chấm mọi thứ 4 sao (≈ hệ số tương quan Pearson).
- **Shrinkage** $\frac{n_{ij}}{n_{ij} + \beta}$ ($n_{ij}$ = số user rate cả hai): hai phim chỉ có 1–2 người chung → tương đồng không đáng tin, bị co về 0.

In [ ]:
resid = np.where(observed, R - (mu + b_u[:, None] + b_i[None, :]), 0.0)


def similarity(Z, beta=5.0):
    """Cosine giữa các CỘT của Z, có shrinkage theo số đồng-quan-sát."""
    norms = np.sqrt((Z**2).sum(0)) + 1e-9
    S = (Z.T @ Z) / np.outer(norms, norms)
    co = (Z != 0).astype(float)
    n_common = co.T @ co
    S = S * n_common / (n_common + beta)
    np.fill_diagonal(S, 0.0)
    return S, n_common


S_item, common_item = similarity(resid)
S_user, _ = similarity(resid.T)
i0 = int(np.argmax(observed.sum(0)))
nbrs = np.argsort(-S_item[i0])[:5]
print(f"phim {i0} ({items.genre[i0]}) — 5 phim tương đồng nhất (theo phần dư rating):")
print(pd.DataFrame({"phim": nbrs, "thể loại": items.genre[nbrs].to_numpy(), "sim": S_item[i0, nbrs].round(3),
                    "số user chung": common_item[i0, nbrs].astype(int)}).to_string(index=False))

In [ ]:
genres = items.genre.to_numpy()
order = np.argsort(genres, kind="stable")
g_sorted = genres[order]
starts = np.r_[0, np.flatnonzero(g_sorted[1:] != g_sorted[:-1]) + 1]
ends = np.r_[starts[1:], len(g_sorted)]
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(S_item[np.ix_(order, order)], cmap="RdBu_r", vmin=-0.3, vmax=0.3)
for s in starts[1:]:
    ax.axhline(s - 0.5, color="k", lw=0.6)
    ax.axvline(s - 0.5, color="k", lw=0.6)
mids = (starts + ends - 1) / 2
ax.set_xticks(mids, g_sorted[starts], rotation=45, ha="right")
ax.set_yticks(mids, g_sorted[starts])
plt.colorbar(im, ax=ax, label="độ tương đồng (phần dư, có shrinkage)")
ax.set_title("Độ tương đồng item–item, sắp theo thể loại")
plt.tight_layout()
plt.show()

Các khối trên đường chéo hơi "đỏ" hơn phần còn lại: phim **cùng thể loại** có phần dư rating tương quan dương — người thích phim kinh dị này thường thích phim kinh dị khác. Nhưng tương quan khá yếu và có cả cặp khác thể loại tương đồng cao: thể loại chỉ là một phần của "gu" người xem — đó là lý do lọc cộng tác học từ hành vi thay vì chỉ dùng thể loại (lọc theo nội dung, bài 01).

## 4. KNN dự đoán rating

$$\hat r_{ui} = b_{ui} + \frac{\sum_{j \in N_k(i; u)}\text{sim}(i, j)\,(r_{uj} - b_{uj})}{\sum_{j \in N_k(i; u)}|\text{sim}(i, j)|}$$
$N_k(i; u)$: $k$ item giống $i$ nhất **trong số item user u đã rate** (item-based). User-based tương tự với $k$ user giống u nhất đã rate item $i$.

In [ ]:
def knn_predict(u, i, k, kind):
    if kind == "item":
        cand = np.flatnonzero(observed[u])
        sims, vals = S_item[i, cand], resid[u, cand]
    else:
        cand = np.flatnonzero(observed[:, i])
        sims, vals = S_user[u, cand], resid[cand, i]
    top = np.argsort(-sims)[:k]
    w = sims[top]
    adj = (w @ vals[top]) / (np.abs(w).sum() + 1e-9) if len(top) else 0.0
    return mu + b_u[u] + b_i[i] + adj


for kind in ["item", "user"]:
    for k in [10, 30]:
        rmse_table[f"{kind}-based KNN, k = {k}"] = rmse(np.array([knn_predict(u, i, k, kind) for u, i in zip(u_te, i_te)]))
pd.Series(rmse_table, name="RMSE test").round(4)

KNN cải thiện thêm so với baseline, nhưng **ít** so với bước từ μ sang $\mu + b_u + b_i$. Trên dữ liệu thật cũng vậy (giải Netflix Prize): thiên lệch giải thích phần lớn phương sai rating; phần cá nhân hoá khó hơn nhiều.

## 5. Top-N từ phản hồi ngầm: "người xem X cũng xem Y"

Cho top-N, câu hỏi không phải "user chấm mấy sao" mà "user có **chọn** xem phim này không". Dùng ma trận **nhị phân** $B_{ui} = 1$ nếu u đã xem i:
- **Item-KNN**: $\text{sim}(i, j) = \cos(B_{:,i}, B_{:,j})$ — hai phim được xem bởi cùng nhóm người; điểm $s_{ui} = \sum_{j \in I_u}\text{sim}(i, j)$.
- **User-KNN**: $k$ user có lịch sử giống u nhất; điểm = tổng có trọng số các phim họ đã xem.

In [ ]:
B = observed.astype(float)
norm_i = np.sqrt(B.sum(0)) + 1e-9
S_bin_item = (B.T @ B) / np.outer(norm_i, norm_i)
np.fill_diagonal(S_bin_item, 0.0)
norm_u = np.sqrt(B.sum(1)) + 1e-9
S_bin_user = (B @ B.T) / np.outer(norm_u, norm_u)
np.fill_diagonal(S_bin_user, 0.0)

seen = {u: set(g.item_id) for u, g in train.groupby("user_id")}
test_ratings = {u: dict(zip(g.item_id, g.rating)) for u, g in test.groupby("user_id")}


def evaluate_topk(score_fn, k=10):
    prec, rec, ndcg = [], [], []
    for u, rated in test_ratings.items():
        liked = [i for i, r in rated.items() if r >= 4]
        if not liked:
            continue
        scores = np.asarray(score_fn(u), float).copy()
        scores[list(seen.get(u, []))] = -np.inf
        top = np.argsort(-scores)[:k]
        prec.append(precision_at_k(top, liked, k))
        rec.append(recall_at_k(top, liked, k))
        ndcg.append(ndcg_at_k(top, {i: r - 3 for i, r in rated.items() if r >= 4}, k))
    return {"precision@10": np.mean(prec), "recall@10": np.mean(rec), "NDCG@10": np.mean(ndcg)}


def user_knn_scores(u, k=30):
    nn = np.argsort(-S_bin_user[u])[:k]
    return S_bin_user[u, nn] @ B[nn]


popularity = B.sum(0)
topn = pd.DataFrame({"popularity (bài 01)": evaluate_topk(lambda u: popularity),
                     "item-KNN (nhị phân)": evaluate_topk(lambda u: S_bin_item @ B[u]),
                     "user-KNN (nhị phân, k = 30)": evaluate_topk(user_knn_scores)}).T.round(4)
topn

Lần đầu tiên vượt popularity: hành vi "ai xem cùng gì" mang thông tin **cá nhân hoá** về việc user sẽ chọn xem phim nào. (Bài 04: vì sao một mô hình dự đoán rating tốt lại có thể xếp hạng tệ.)

In [ ]:
print(f"'Người xem phim {i0} ({items.genre[i0]}) cũng xem':")
also = np.argsort(-S_bin_item[i0])[:5]
print(pd.DataFrame({"phim": also, "thể loại": items.genre[also].to_numpy(), "cosine": S_bin_item[i0, also].round(3),
                    "số người xem": B[:, also].sum(0).astype(int)}).to_string(index=False))

Danh sách "cũng xem" toàn phim **rất phổ biến** (hơn 90 người xem, trong khi trung vị chỉ vài chục): phim nào nhiều người xem cũng hay "đi cùng" mọi phim khác — **thiên lệch phổ biến** của cosine trên dữ liệu nhị phân. Các hệ thống thật thường chia thêm cho độ phổ biến (vd lift, hoặc mũ $\alpha$ trên chuẩn) để gợi ý bớt hiển nhiên.

## 6. Vấn đề thực tế

| Vấn đề | Mô tả | Hướng xử lý |
|---|---|---|
| Thưa | ít user chung → tương đồng nhiễu | shrinkage, giảm chiều (bài 03) |
| Cold start | user/item mới không có tương tác | content-based, popularity, hỏi sở thích ban đầu |
| Quy mô | ma trận tương đồng item $O(n_{\text{item}}^2)$, user $O(n_{\text{user}}^2)$ | item-based (item ổn định hơn user, tính trước), ANN, chỉ giữ top-k láng giềng |
| Thiên lệch phổ biến | item phổ biến xuất hiện trong mọi láng giềng | chuẩn hoá, metric độ phủ/mới lạ (bài 04) |

Item-based CF là nền tảng của hệ thống gợi ý Amazon thời kỳ đầu (Linden, Smith & York, 2003): danh sách "item tương tự" tính offline, tra cứu nhanh khi phục vụ.

## 🧠 Kiểm tra nhanh

1. Baseline predictor $\mu + b_u + b_i$ bắt được gì? Vì sao cần $\lambda$?
2. Vì sao tính tương đồng trên phần dư thay vì rating thô? Shrinkage để làm gì?
3. Viết công thức dự đoán item-based KNN.
4. Vì sao item-based thường được chọn trong production hơn user-based?
5. Vì sao top-N dùng ma trận nhị phân "đã xem" thay vì rating?

## 📌 Tóm tắt

- CF dùng hành vi người khác; thiên lệch user/item giải thích phần lớn biến động rating.
- KNN: tương đồng (phần dư, shrinkage) → trung bình có trọng số của láng giềng.
- Top-N từ phản hồi ngầm: item/user-KNN nhị phân vượt popularity; cold start, thưa, quy mô là thách thức chính.

## 📚 Tài liệu tham khảo

- Koren (2008) — *Factorization Meets the Neighborhood* (baseline predictor, shrinkage)
- Sarwar, Karypis, Konstan & Riedl (2001) — *Item-based Collaborative Filtering Recommendation Algorithms*
- Linden, Smith & York (2003) — *Amazon.com Recommendations: Item-to-Item Collaborative Filtering*